# Практическая работа 4. Покупки в конце сессии

Вариант 8

Объединяем оба журнала. Для одного пользователя перерыв **больше 30 минут** открывает новую сессию. Покупающей считаем сессию, в которой последнее событие — `purchase`.

In [1]:
import pandas as pd
from shop_data import read_events

events = read_events().drop_duplicates("event_id")
events = events.sort_values(["user_id", "event_time", "event_id"])
pause = events.groupby("user_id")["event_time"].diff()
new_session = pause.isna() | (pause > pd.Timedelta(minutes=30))
events["session_id"] = new_session.groupby(events["user_id"]).cumsum()

### 1. Последнее событие сессии

При совпадении времени порядок определяет идентификатор события.

In [2]:
sessions = events.groupby(["user_id", "session_id"]).agg(
    started=("event_time", "first"),
    ended=("event_time", "last"),
    last_event=("event_type", "last"),
)
sessions["ends_with_purchase"] = sessions["last_event"].eq("purchase")
print("Всего сессий:", len(sessions))
sessions.head(10)

Всего сессий: 3950


started               ended last_event  \
user_id session_id                                                      
1001    1          2024-01-01 06:25:31 2024-01-01 06:25:31   purchase   
        2          2024-01-05 07:12:56 2024-01-05 07:12:56  page_view   
        3          2024-01-08 15:40:51 2024-01-08 15:40:51      click   
        4          2024-01-09 06:39:26 2024-01-09 06:39:26     scroll   
        5          2024-01-15 00:00:38 2024-01-15 00:00:38  page_view   
        6          2024-01-20 00:29:26 2024-01-20 00:29:26  page_view   
        7          2024-01-24 08:41:28 2024-01-24 08:41:28  page_view   
        8          2024-01-24 11:54:49 2024-01-24 11:54:49      click   
        9          2024-01-25 17:54:45 2024-01-25 17:54:45   purchase   
        10         2024-01-26 17:20:09 2024-01-26 17:20:09  page_view   

                    ends_with_purchase  
user_id session_id                      
1001    1                         True  
        2                        False  
        3                        False  
        4                        False  
        5                        False  
        6                        False  
        7                        False  
        8                        False  
        9                         True  
        10                       False

### 2. Доля покупающих сессий у каждого пользователя

In [3]:
buyers = sessions.groupby("user_id")["ends_with_purchase"].agg(
    sessions="size", buying_sessions="sum", purchase_share="mean"
)
buyers.sort_values("purchase_share", ascending=False)

,sessions,buying_sessions,purchase_share
user_id,,,
1003,39,16,0.410256
1004,35,14,0.400000
1058,45,16,0.355556
1076,44,14,0.318182
1048,29,9,0.310345
...,...,...,...
1085,41,4,0.097561
1082,24,2,0.083333
1037,39,3,0.076923


### 3. Пользователи с наибольшей долей

Выводим всех пользователей, достигших максимума.

In [4]:
leaders = buyers[buyers["purchase_share"] == buyers["purchase_share"].max()]
leaders.round({"purchase_share": 4})

,sessions,buying_sessions,purchase_share
user_id,,,
1003,39,16,0.4103


Наибольшая доля у пользователя 1003. Покупкой завершились 16 из 39 сессий (41,03 %).